In [1]:
import torch

import scipy.sparse as sp
import numpy as np
import plotly.express as px

import pandas as pd

In [2]:
df = pd.concat([
    pd.read_csv(f, index_col=0) for f in ["benchmark_results_norm_none_200_iters.csv",
                             "benchmark_results_norm_left_200_iters.csv",
                             "benchmark_results_norm_right_200_iters.csv",
                             "benchmark_results_norm_both_200_iters.csv"]
])
df.loc[df["graph"] == "synthetic", "graph"] = df.loc[df["graph"] == "synthetic", "graph"] + "_" + (df.loc[df["graph"] == "synthetic", "num_nodes"] // 1000).astype(str) + "k_nodes"

In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

norms = df['norm'].unique()
num_norms = len(norms)

cols = 2
rows = (num_norms + cols - 1) // cols


feature_dims = sorted(df['feature_dim'].unique())

# Pick a color palette (e.g. Plotly qualitative palette)
palette = px.colors.qualitative.Plotly

# Create a mapping from feature_dim to color, cycle palette if needed
color_map = {dim: palette[i % len(palette)] for i, dim in enumerate(feature_dims)}



fig = make_subplots(rows=rows, cols=cols, subplot_titles=[f"Norm = {n}" for n in norms])

for i, norm in enumerate(norms):
    df_sub = df[df['norm'] == norm]
    feature_dims = df_sub['feature_dim'].unique()
    
    row = i // cols + 1
    col = i % cols + 1

    for dim in feature_dims:
        df_dim = df_sub[df_sub['feature_dim'] == dim]

        hover_texts = [f"Avg degree: {v:.2f}" for v in df_dim['avg_degree']]

        fig.add_trace(
            go.Bar(
                x=df_dim['graph'],
                y=df_dim['speedup'],
                name=f"dim {dim}",
                legendgroup=str(dim),
                showlegend=(i == 0),
                hovertext=hover_texts,
                hoverinfo="text+y+x",
                marker_color=color_map[dim],  # Assign color here
            ),
            row=row, col=col
        )

    fig.update_xaxes(title_text="Dataset", row=row, col=col)
    fig.update_yaxes(title_text="Speedup", row=row, col=col)
    
    # Add horizontal line y=1 in this subplot
    fig.add_hline(y=1, line_dash="dash", line_color="black", row=row, col=col, annotation_text="no speedup", annotation_position="bottom right"
)

fig.update_layout(
    height=500 * rows,
    width=1000,
    title_text="Speedup per Dataset by Norm and Feature Dim",
    barmode='group'
)

fig.show()


In [4]:
norms = df['norm'].unique()
num_norms = len(norms)

cols = 2
rows = (num_norms + cols - 1) // cols


feature_dims = sorted(df['feature_dim'].unique())

# Pick a color palette (e.g. Plotly qualitative palette)
palette = px.colors.qualitative.Plotly

# Create a mapping from feature_dim to color, cycle palette if needed
color_map = {dim: palette[i % len(palette)] for i, dim in enumerate(feature_dims)}



fig = make_subplots(rows=rows, cols=cols, subplot_titles=[f"Norm = {n}" for n in norms])

for i, norm in enumerate(norms):
    df_sub = df[df['norm'] == norm]
    feature_dims = df_sub['feature_dim'].unique()
    
    row = i // cols + 1
    col = i % cols + 1

    for dim in feature_dims:
        df_dim = df_sub[df_sub['feature_dim'] == dim]

        hover_texts = [f"Avg degree: {v:.2f}" for v in df_dim['avg_degree']]

        fig.add_trace(
            go.Bar(
                x=df_dim['graph'],
                y=df_dim['max_diff'],
                name=f"dim {dim}",
                legendgroup=str(dim),
                showlegend=(i == 0),
                hovertext=hover_texts,
                hoverinfo="text+y+x",
                marker_color=color_map[dim],  # Assign color here
            ),
            row=row, col=col
        )

    fig.update_xaxes(title_text="Dataset", row=row, col=col)
    fig.update_yaxes(title_text="Max diff", row=row, col=col)
    
    # Add horizontal line y=1 in this subplot
    # fig.add_hline(y=1, line_dash="dash", line_color="black", row=row, col=col, annotation_text="speedup < 1", annotation_position="bottom right")

fig.update_layout(
    height=500 * rows,
    width=1000,
    title_text="Max diff per Dataset by Norm and Feature Dim",
    barmode='group'
)

fig.show()